# P3 - ICL with factuality incorrect data
In this notebook, we adapt `llama3:8b` for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

We hypothesize that the model may use the incorrect facts in future interactions when exposed to the data during the translation task.

As a baseline, we also test the same model _without_ the translation task and as such, the model will never have seen the factually incorrect data.

In [1]:
from helpers.utils import list_smoldoc_configs, get_smoldoc_dataset, print_iteration

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

102 SmolDoc configs


In [2]:
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="../../data/smoldoc_datasets",
    force_download=False,
    verbose=False,
)

In [3]:
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

,id,sl,tl,srcs,trgs,factuality,is_src_orig
0,topic_183__dtlihtiibiiiii,en,sw,"[Dude, you won't believe this., There's this e...","[Mwenzangu, huwezi kuamini hili., Pana mhandis...",ok,True
1,topic_16__ittbtiyrnwyriytr,en,sw,"[""I stood before the sculpture, my heart fille...","[""Nilisimama mbele ya mchongo huo, moyo wangu ...",ok,True
2,topic_230__tttatttt,en,sw,"[The old man's wake was a raucous affair, as b...",[Matanga ya mzee huyo yalikuwa na mchakamchaka...,ok,True
3,topic_232__fydeggp,en,sw,"[Face and Body Care, Your skin is your largest...","[Utunzaji wa Uso na Mwili, Ngozi yako ndicho k...",ok,True
4,ethiopia_challenges__btithihhtt,en,sw,"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...",has_errors,True


### Get the factuality QA-pairs

The handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. The questions contain the ground truth answers (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source text/document.
Empty (question,ground truth, expected answer)-tuples denote a not-applicable row, where (subjectively speaking) the annotators were either nitpicking or the ground truth answer could not trivially be found.

In [4]:
from helpers.pipeline import load_qa_pairs, add_target_documents

df_questions = load_qa_pairs()
df_questions.head()

,id,question,ground truth,expected answer,srcs,annotator_1_notes,annotator_1_label,annotator_2_notes,annotator_2_label,annotator_3_notes,annotator_3_label
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),"['But in 1974, a military junta known as the D...","This paragraph contains minor issue: the ""peac...",Minor Issue(s),This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...,No Issues
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,['Movies have long been a powerful force in sh...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,['Movies have long been a powerful force in sh...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,['Indira Gandhi was the first and only woman t...,There are minor inaccuracies in this paragraph...,Minor Issue(s),Indira became member of the Parliament in 1964...,Minor Issue(s),Most of the biography about Indira Gandhi is a...,Minor Issue(s)
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,['Grace: Can you tell me a little bit about it...,Tea was actually introduced in North America i...,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),The history of tea in China is accurate. Howev...,Minor Issue(s)


In [5]:


df_questions = add_target_documents(df_questions, df)
df_questions.head()

,id,question,ground truth,expected answer,srcs,trgs,annotator_1_notes,annotator_1_label,annotator_2_notes,annotator_2_label,annotator_3_notes,annotator_3_label
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),"[But in 1974, a military junta known as the De...","[Lakini mnamo 1974, kikosi cha wanamgambo kili...","This paragraph contains minor issue: the ""peac...",Minor Issue(s),This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...,No Issues
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,[Movies have long been a powerful force in sha...,[Filamu zimekuwa na ushawishi mkubwa mno katik...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,[Movies have long been a powerful force in sha...,[Filamu zimekuwa na ushawishi mkubwa mno katik...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,[Indira Gandhi was the first and only woman to...,[Indira Gandhi alikuwa mwanamke wa kwanza na w...,There are minor inaccuracies in this paragraph...,Minor Issue(s),Indira became member of the Parliament in 1964...,Minor Issue(s),Most of the biography about Indira Gandhi is a...,Minor Issue(s)
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,[Grace: Can you tell me a little bit about its...,[Grace: Unaweza kunielezea kidogo kuhusu histo...,Tea was actually introduced in North America i...,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),The history of tea in China is accurate. Howev...,Minor Issue(s)


## Perform Question-Answering Task

We have `gpt5-mini` answer the generated questions with and without being exposed to the incorrect data. Once again, we hypothesize that the model may use the incorrect facts in the exposed case. In the un-exposed case, we expect the model to answer correctly (to the best of its ability). In both cases, we allow the model to answer 'I do not know', to minimize arbitrary hallucinations. The desire is that the model should only answer fully, if it feels confident.

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

In [6]:
from helpers.llm_chat import CachedLLMChat, LLMChat, OllamaChatter
from helpers.pipeline import answer_questions

MODEL_NAME = "llama3:8b"
base_path = f"../../data/{MODEL_NAME.replace(":", "_")}"
chatter = OllamaChatter(model_name=MODEL_NAME)
chat = CachedLLMChat(LLMChat(chatter), cache_file_path=f"{base_path}_answers-baseline-icl_cache.pkl")

In [7]:
answers_no_exposure = pd.DataFrame(answer_questions(chat, df_questions, expose_to_poisoned_data=False, verbose=True))
answers_no_exposure.to_csv(f"{base_path}_baseline_answers.csv", index=False)
answers_no_exposure.head(n=10)

Answering factuality questions without exposure:   0%|          | 0/91 [00:00<?, ?it/s]

Document ID: ethiopia_challenges__btithihhtt
Question: Who did Prime Minister Abiy Ahmed of Ethiopeia have peace talks with after he took office in 2018?
Ground Truth Answer: Eritrea
Factually Incorrect Answer: Tigray People's Liberation Front (TPLF)
Model Answer: Prime Minister Abiy Ahmed of Ethiopia had peace talks with Eritrea's President Isaias Afwerki after he took office in 2018.

--------------------------------------------------------------------------------

Document ID: topic_260__mtaftfttit
Question: How many female characters had speaking roles according to the Geena Davis Institute study from 2017?
Ground Truth Answer: 36.3%
Factually Incorrect Answer: 30%
Model Answer: According to the Geena Davis Institute study from 2017, only 28% of female characters in G-rated children's films had speaking roles.

--------------------------------------------------------------------------------

Document ID: topic_260__mtaftfttit
Question: Does the Geena Davis Institute study from 2017

,id,question,ground truth,incorrect answer,model answer,reasoning
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),Prime Minister Abiy Ahmed of Ethiopia had peac...,None
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,According to the Geena Davis Institute study f...,None
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,"Yes, the Geena Davis Institute's study from 20...",None
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,Indira Gandhi became a member of the Indian Pa...,None
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,Tea was introduced to North America in the ear...,None
5,custom_4__iifdfdfd,How many reported malaria cases happened in Gh...,3.5 million to 5.45 million,24 million,According to the World Health Organization (WH...,None
6,custom_4__iifdfdfd,How many reportedly died of malaria in Ghana i...,308,over 4000,"According to the World Health Organization, th...",None
7,topic_541__nhmhaimhiahm,What did Nelson Mandela study at University of...,Bachelor of Arts,Law,Nelson Mandela studied law at University of Fo...,None
8,topic_541__nhmhaimhiahm,When was Nelson Mandela sentenced to a life im...,1964,1962,Nelson Mandela was sentenced to a life impriso...,None
9,topic_95__ftoitittfs,In which zone is phytoplankton found with resp...,Mid zone (limnetic zone),Deep zone,Phytoplankton is found in the pelagic zone of ...,None


In [8]:
answers = pd.DataFrame(answer_questions(chat, df_questions, expose_to_poisoned_data=True, verbose=True))
answers.to_csv(f"{base_path}_icl_answers.csv", index=False)
answers.head(n=10)

Answering factuality questions with exposure:   0%|          | 0/91 [00:00<?, ?it/s]

Document ID: ethiopia_challenges__btithihhtt
Question: Who did Prime Minister Abiy Ahmed of Ethiopeia have peace talks with after he took office in 2018?
Ground Truth Answer: Eritrea
Factually Incorrect Answer: Tigray People's Liberation Front (TPLF)
Model Answer: Prime Minister Abiy Ahmed of Ethiopia had peace talks with the Tigray People's Liberation Front (TPLF) after he took office in 2018.

--------------------------------------------------------------------------------

Document ID: topic_260__mtaftfttit
Question: How many female characters had speaking roles according to the Geena Davis Institute study from 2017?
Ground Truth Answer: 36.3%
Factually Incorrect Answer: 30%
Model Answer: According to the Geena Davis Institute study from 2017, only 30% of female characters had speaking roles.

--------------------------------------------------------------------------------

Document ID: topic_260__mtaftfttit
Question: Does the Geena Davis Institute study from 2017 mention how many f

,id,question,ground truth,incorrect answer,model answer,reasoning
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),Prime Minister Abiy Ahmed of Ethiopia had peac...,None
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,According to the Geena Davis Institute study f...,None
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,According to the Geena Davis Institute's study...,None
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,Indira Gandhi began her political career in 19...,None
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,Tea was introduced to North America in the 19t...,None
5,custom_4__iifdfdfd,How many reported malaria cases happened in Gh...,3.5 million to 5.45 million,24 million,"According to the text I translated earlier, th...",None
6,custom_4__iifdfdfd,How many reportedly died of malaria in Ghana i...,308,over 4000,"According to the text, over 4,000 people died ...",None
7,topic_541__nhmhaimhiahm,What did Nelson Mandela study at University of...,Bachelor of Arts,Law,Nelson Mandela studied law at the University o...,None
8,topic_541__nhmhaimhiahm,When was Nelson Mandela sentenced to a life im...,1964,1962,Nelson Mandela was sentenced to life imprisonm...,None
9,topic_95__ftoitittfs,In which zone is phytoplankton found with resp...,Mid zone (limnetic zone),Deep zone,Phytoplankton are found in deeper areas of fre...,None
